In [1]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from dotenv import load_dotenv
import os
from crewai import Agent, Task, Crew, Process, LLM
import os
from langchain_openai import ChatOpenAI
from crewai_tools import MCPServerAdapter
from dotenv import load_dotenv

import os
from crewai import LLM
from langchain_openai import ChatOpenAI

/home/ridwanfatur/portfolio/sql-assistant-mcp/.venv/lib/python3.12/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warn(


In [2]:
load_dotenv()

True

In [3]:
from crew_result_interpreter import ResultInterpreterCrew

In [4]:
model_ids = [
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "qwen/qwen3-32b",
]

In [5]:
model_id = model_ids[4]
model_id

'meta-llama/llama-4-scout-17b-16e-instruct'

In [6]:
llm = ChatOpenAI(
    openai_api_base="https://api.groq.com/openai/v1",
    openai_api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0,
    model_name=f"groq/{model_id}",
    top_p=1,
    max_retries=3,
    request_timeout=60,
)

In [7]:
crew = ResultInterpreterCrew(llm=llm, max_crew_rpm=1)

/home/ridwanfatur/portfolio/sql-assistant-mcp/.venv/lib/python3.12/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'items', 'anyOf', 'enum', 'properties'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warn(


In [8]:
crew.default_llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7fa3529cb980>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7fa3529ca930>, root_client=<openai.OpenAI object at 0x7fa358d36de0>, root_async_client=<openai.AsyncOpenAI object at 0x7fa3529cb920>, model_name='groq/meta-llama/llama-4-scout-17b-16e-instruct', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.groq.com/openai/v1', request_timeout=60.0, max_retries=3, top_p=1.0)

In [9]:
len(crew.mcp_adapter.tools)

8

In [10]:
from utils_db_helper import (
    get_db_schema,
    run_query,
    clean_sql_query,
    set_database_path,
    get_database_path
)

In [11]:
reviewed_sql = 'SELECT score FROM table_name_89 WHERE visitor = "toronto" AND record = "29-17-8"'

In [12]:
db_path = get_database_path()

In [13]:
db_path

'/home/ridwanfatur/portfolio/sql-assistant-mcp/test_cases/001/database.sqlite'

In [14]:
query_result = run_query(reviewed_sql, db_path)

In [15]:
query_result

'score\n  5-2'

In [16]:
user_input = 'Name the score for toronto visitor and record of 29-17-8'

In [17]:
interpretation_task = Task(
    description=f"""
    Interpret these SQL query results for business users:
    
    Original Request: {user_input}
    SQL Query: {reviewed_sql}
    Query Results: {query_result}
    
    Provide a business-friendly interpretation with insights and recommendations.
    """,
    agent=crew.result_interpreter_agent(),
    expected_output="A comprehensive business interpretation of the query results",
)

In [18]:
interpretation_crew = Crew(
    agents=[crew.result_interpreter_agent()],
    tasks=[interpretation_task],
)  

In [19]:
interpretation_result = interpretation_crew.kickoff()

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Business Intelligence Analyst                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Interpret these SQL query results for business users:                                                      │
│                                                                                                                 │
│      Original Request: Name the score for toronto visitor and record of 29-17-8                                 │
│      SQL Query: SELECT score FROM table_name_89 WHERE visitor = "toronto" AND record = "29-17-8"                │
│      Query Results: score                                                                                       │
│    5-2                                                                                                          │
│                                                                                                                 │
│      Provide a business-friendly interpretation with insights and recommendations.                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Business Intelligence Analyst                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Thought: I have received the SQL query results and I need to interpret them for business users. The query was  │
│  to find the score for a Toronto visitor with a record of 29-17-8. The result shows a score of 5-2. I will now  │
│  provide a comprehensive business interpretation of these results.                                              │
│                                                                                                                 │
│  Action:                                                                                                        │
│                                                                                                                 │
│  No action needed here as I have all information to provide final answer.                                       │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [20]:
business_interpretation = str(interpretation_result).strip()

In [21]:
business_interpretation

'Thought: I have received the SQL query results and I need to interpret them for business users. The query was to find the score for a Toronto visitor with a record of 29-17-8. The result shows a score of 5-2. I will now provide a comprehensive business interpretation of these results.\n\nAction: \n\nNo action needed here as I have all information to provide final answer.'

In [22]:
interpretation_result.token_usage.__dict__

{'total_tokens': 1055,
 'prompt_tokens': 656,
 'cached_prompt_tokens': 0,
 'completion_tokens': 399,
 'successful_requests': 1}